In [3]:
import json

# 读取 JSON 文件
def load_json_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

# 解析对话内容
def parse_conversations(data):
    for conversation in data:
        print(f"Conversation ID: {conversation['id']}")
        for turn in conversation['conversations']:
            speaker = turn['from']
            message = turn['value']
            print(f"{speaker}: {message}")
        print("-" * 40)  # 分隔线

# 主函数
def main():
    file_path = 'sharegpt_data.json'  # 替换为你的 JSON 文件路径
    data = load_json_file(file_path)
    parse_conversations(data)

In [4]:
data = load_json_file("./data_news_300/eval.json")

In [40]:
file_path = "../data/sharegpt_eval_post_cot.json"
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
label_list = []
for conversation in data:
    for turn in conversation['conversations']:
        if turn['from'] == 'gpt':
            label_list.append(turn['value'].split("</think>")[1].strip())

In [1]:
import os
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

def pr(y_true, y_pred):
    precision = precision_score(y_true, y_pred, average=None)
    recall = recall_score(y_true, y_pred, average=None)
    f1 = f1_score(y_true, y_pred, average=None)
    print("base", "p:", precision, "r:", recall, "f1:", f1)

def all_pr(y_true, y_pred):
    y_true = [1 if x == 2 else x for x in y_true]
    y_pred = [1 if x == 2 else x for x in y_pred]

    precision_micro = precision_score(y_true, y_pred, average=None)
    recall_micro = recall_score(y_true, y_pred, average=None)
    f1_micro = f1_score(y_true, y_pred, average=None)
    print("merge p:", precision_micro, "r:", recall_micro, "f1:", f1_micro)

In [4]:
def label_process(label_list):
    result_list = []
    for i in label_list:
        result = -1
        if "底线" in i:
            result = 2
        elif "擦边" in i:
            result = 1
        elif "无" in i:
            result = 0
        result_list.append(result)
    return result_list

file_path = "../data/sharegpt_eval_post_cot.json"
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
label_list = []
for conversation in data:
    for turn in conversation['conversations']:
        if turn['from'] == 'gpt':
            label_list.append(turn['value'].split("</think>")[1].strip())
result_list = []
for i in open("./data/result_sharegpt_eval_post_cot.json"):
    case = json.loads(i.strip())
    result = ""
    try:
        result = case['output'].split("</think>\n\n")[1]
    except:
        pass
    result_list.append(result)

In [5]:
grownd_ture = label_process(label_list)
pred = label_process(result_list)

data = pd.DataFrame()
data["grownd_ture"] = grownd_ture
data["pred"] = pred

print("error num: ", (data["pred"] == -1).sum())
data = data[data["pred"] != -1]
pr(data["grownd_ture"], data["pred"])
all_pr(data["grownd_ture"], data["pred"])

error num:  9
base p: [0.95141607 0.84395716 0.93173821] r: [0.96858884 0.90092542 0.8162762 ] f1: [0.95992566 0.87151132 0.87019389]
merge p: [0.95141607 0.96037847] r: [0.96858884 0.93899971] f1: [0.95992566 0.94956878]


In [6]:
file_path = "../data/sharegpt_eval_post_cot.json"
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
label_list = []
for conversation in data:
    for turn in conversation['conversations']:
        if turn['from'] == 'gpt':
            label_list.append(turn['value'].split("</think>")[1].strip())
result_list = []
with open("./data/result_sharegpt_eval_post_cot_vllm.json", 'r', encoding='utf-8') as f:
    data_pred = json.load(f)
    for case in data_pred:
        result = ""
        try:
            result = case['output'].split("</think>\n\n")[1]
        except:
            pass
        result_list.append(result)

grownd_ture = label_process(label_list)
pred = label_process(result_list)

data = pd.DataFrame()
data["grownd_ture"] = grownd_ture
data["pred"] = pred

print("error num: ", (data["pred"] == -1).sum())
data = data[data["pred"] != -1]
pr(data["grownd_ture"], data["pred"])
all_pr(data["grownd_ture"], data["pred"])

error num:  0
base p: [0.98331402 0.96690568 0.98132005] r: [0.99344416 0.95375408 0.96984615] f1: [0.98835313 0.96028485 0.97554937]
merge p: [0.98331402 0.99181047] r: [0.99344416 0.97920878] f1: [0.98835313 0.98546934]


In [7]:
file_path = "./data/zeus_benchmark.txt"
data = pd.read_csv(file_path, sep="\t")
data["grownd_ture"] = data["tgt"].apply(lambda x: int(x.split("[SEP]")[0]))

result_list = []
with open("./data/result_benchmark_vllm.json", 'r', encoding='utf-8') as f:
    data_pred = json.load(f)
    for case in data_pred:
        result = ""
        try:
            result = case['output'].split("</think>")[1].strip()
        except:
            pass
        result_list.append(result)

pred = label_process(result_list)
data["pred"] = pred

print("error num: ", (data["pred"] == -1).sum())
data = data[data["pred"] != -1]
pr(data["grownd_ture"], data["pred"])
all_pr(data["grownd_ture"], data["pred"])

error num:  0
base p: [0.93915858 0.80649718 0.74404762] r: [0.95116355 0.78112175 0.71022727] f1: [0.94512294 0.79360667 0.72674419]
merge p: [0.93915858 0.85727969] r: [0.95116355 0.82640813] f1: [0.94512294 0.84156088]


In [77]:
i['output']

'分析：未发现风险点\n</think>\n\n无风险'

In [48]:
!pip install --upgrade pip
!pip install scikit-learn

Looking in indexes: http://pip.baidu-int.com/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 50.3 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 22.3.1
    Uninstalling pip-22.3.1:
      Successfully uninstalled pip-22.3.1
Looking in indexes: http://pip.baidu-int.com/simple/


In [ ]:
10-11:30 , 1-1.5
2-5:30, 3.5
7-8:30,10, 1.5-3 

In [83]:
6-8

8.0

In [7]:
for conversation in data:
    break
    for turn in conversation['conversations']:
        speaker = turn['from']
        message = turn['value']
        print(f"{speaker}: {message}")
    print("-" * 40)  # 分隔线

In [9]:
for turn in conversation['conversations']:
    speaker = turn['from']
    message = turn['value']
    print(f"{speaker}: {message}")

human: 新闻分类：大宋，据说这个游戏已经倒闭了 但是武将的游戏，还在继续
gpt: <think>
首先，我需要分析这条新闻标题：“大宋，据说这个游戏已经倒闭了 但是武将的游戏，还在继续”。标题中提到了“大宋”，这指的是宋朝，而宋朝属于中国的国内历史时期。接着，标题讨论了关于一款游戏的倒闭情况，以及武将游戏继续发展的信息。

根据新闻种类的定义，“国内”指的是中国的内部事务，包括地方新闻、社会事件等。这里的内容涉及到一个游戏的倒闭和另一个游戏的继续，属于游戏行业内的动态，但它们都与国内的『娱乐』领域相关，因为游戏属于娱乐产业。

虽然 technological (科技) 也是一个可能的分类，但标题更侧重于娱乐方面的动态，而不是科技新闻。因此，恰当的分类应该是“娱乐”。

总结一下，标题主要讨论的是国内的娱乐新闻，涉及游戏行业的动态变化，因此将其归类为“娱乐”。
</think>

娱乐


In [ ]:
sharegpt_data = []
user_message = {"from": "user", "value": prompt}
assistant_message = {"from": "gpt", "value": response}

sharegpt_data.append({
    "id": conv_id,
    "conversations": [user_message, assistant_message]
})

In [10]:
import pandas as pd

In [15]:
data = pd.read_csv("/root/paddlejob/workspace/env_run/baidu/fengkong/aigc-risk-management/llm_risk_model/model_zeus/ernie3_zeus_train/ernie3_zeus/workspace/data/fc_train_0212_has_industry.txt", sep="\t")

In [17]:
result = []
for i in range(len(data)):
    case = data.iloc[i]

    src = case["src"]
    tgt = case["tgt"].split("[SEP]")[1]

    reason = ""
    label = tgt.split("，")[0]
    if label == "无风险":
        reason = "内容未发现风险"
    else:
        tgt.split("，")[1]



0        凤巢平台，行业：生活服务-婚恋相亲，广告：女人脱精光直播app-正版下载-免费使用。请推断以...
1        凤巢平台，行业：软件-实用工具，广告：哪些因素会影响有效沟通?看完这篇你就知道了。请推断以上...
2        凤巢平台，行业：软件-实用工具，广告：一键p掉衣服的软件,只需一个步骤。请推断以上广告是否存...
3        凤巢平台，行业：软件-实用工具，广告：欧美性生活教育片-无限制兔费欧美性生活教育片-观看全网...
4        凤巢平台，行业：软件-社交通讯，广告：元宇宙 创新 全国通用 可用。请推断以上广告是否存在色...
                               ...                        
12331    凤巢平台，行业：软件-实用工具，广告：苹果删掉微信后聊天记录怎么恢复_简单恢复方法。请推断以...
12332    凤巢平台，行业：软件-实用工具，广告：{关键词}{组织架构图制作软件},简单好用的组织架构图...
12333    凤巢平台，行业：软件-社交通讯，广告：午夜高清福利视频完整版-劲爆热舞-刺激不断。请推断以上...
12334    凤巢平台，行业：软件-社交通讯，广告：呼市满都海公园相亲角-成家-立业-成人交友平台。请推断...
12335    凤巢平台，行业：软件-社交通讯，广告：大同 -成人MM直播app大秀,害羞男士禁止下载,。请...
Name: src, Length: 12336, dtype: object

In [18]:
data

,qid,src,tgt
0,0,凤巢平台，行业：生活服务-婚恋相亲，广告：女人脱精光直播app-正版下载-免费使用。请推断以...,"2[SEP]底线色情，风险点：女人脱精光直播app, 类型：色情资源, 原因：女人脱精光直播..."
1,1,凤巢平台，行业：软件-实用工具，广告：哪些因素会影响有效沟通?看完这篇你就知道了。请推断以上...,0[SEP]无风险，原因：
2,2,"凤巢平台，行业：软件-实用工具，广告：一键p掉衣服的软件,只需一个步骤。请推断以上广告是否存...","2[SEP]底线色情，风险点：一键p掉衣服的软件, 类型：借助技术实现色情功能, 原因：一键..."
3,3,凤巢平台，行业：软件-实用工具，广告：欧美性生活教育片-无限制兔费欧美性生活教育片-观看全网...,"2[SEP]底线色情，风险点：无限制兔费欧美性生活教育片, 类型：色情资源, 原因：无限制兔..."
4,4,凤巢平台，行业：软件-社交通讯，广告：元宇宙 创新 全国通用 可用。请推断以上广告是否存在色...,0[SEP]无风险，原因：
...,...,...,...
12331,12331,凤巢平台，行业：软件-实用工具，广告：苹果删掉微信后聊天记录怎么恢复_简单恢复方法。请推断以...,0[SEP]无风险，原因：
12332,12332,"凤巢平台，行业：软件-实用工具，广告：{关键词}{组织架构图制作软件},简单好用的组织架构图...",0[SEP]无风险，原因：
12333,12333,凤巢平台，行业：软件-社交通讯，广告：午夜高清福利视频完整版-劲爆热舞-刺激不断。请推断以上...,0[SEP]无风险，原因：午夜高清福利视频是正常视频，无色情低俗含义
12334,12334,凤巢平台，行业：软件-社交通讯，广告：呼市满都海公园相亲角-成家-立业-成人交友平台。请推断...,0[SEP]无风险，原因：


-1

In [13]:
file_path = "../data/sharegpt_eval_post_cot.json"
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)
data_list = []
for conversation in data:
    prompt = response = ""
    for turn in conversation['conversations']:
        if turn['from'] == 'human':
            prompt = turn['value']
        if turn['from'] == 'gpt':
            response = turn['value']
    data_list.append([prompt, response])
    

In [15]:
data_list

[['凤巢平台，行业：软件-实用工具，广告：日本女人洗澡的视频 挠日本美女脚心视频 精彩不断高清影视全集 让您畅享精彩使用体验 高清影视聚集地 免费观看。请推断以上广告是否存在色情低俗风险',
  '<think> 分析：日本女人洗澡的视频 挠日本美女脚心视频直接涉及色情资源，风险点：日本女人洗澡的视频 挠日本美女脚心视频，类型：色情资源</think> 底线色情'],
 ['凤巢平台，行业：软件-实用工具，广告：成人在线视频-点击进入。请推断以上广告是否存在色情低俗风险',
  '<think> 分析：成人表示固定年龄范围，暗示未成年人不能看，涉及低俗资源，风险点：成人在线视频，类型：低俗资源</think> 低俗擦边'],
 ['凤巢平台，行业：软件-社交通讯，广告：[18以下禁入] 男女交性app [开放-刺激] 约聊,同城美女。请推断以上广告是否存在色情低俗风险',
  '<think> 分析：男女交性app是性交的意思，属于性行为与性器官描述，风险点：男女交性app，类型：性行为与性器官描述</think> 底线色情'],
 ['凤巢平台，行业：软件-实用工具，广告：掌上华医旧版本-简单好用的手机APP。请推断以上广告是否存在色情低俗风险',
  '<think> 分析：未发现风险点</think> 无风险'],
 ['凤巢平台，行业：软件-实用工具，广告：红单团队具备高效的生产线,平台拥有数千名赛事分析专家,提供精准信息服务, 赛事分析就看疯狂红单,实时为您提供及时情报信息。请推断以上广告是否存在色情低俗风险',
  '<think> 分析：红单团队，无色情擦边内容</think> 无风险'],
 ['凤巢平台，行业：软件-社交通讯，广告：交友软件app推荐-害羞男士禁止下载。请推断以上广告是否存在色情低俗风险',
  '<think> 分析：害羞男士禁止下载涉及隐晦表达有不能直接看到内容，属于低俗交友，风险点：交友软件app推荐-害羞男士禁止下载，类型：低俗交友</think> 低俗擦边'],
 ['凤巢平台，行业：软件-实用工具，广告：{关键词}{恢复和已删除人的微信聊天记录}-只需三个步骤, 1.恢复微信误删记录2.恢复误删好友3.找回微信记录4.微信好友找回5, .微信聊天恢复6.其他文件恢复,支持一键恢复找回,用这个方法再也不用担心找不回啦!。